# PRISMA 2020 flow diagram — CNKI systematic review (96 included articles)

The counts are **computed from the screening workbook** (`screening_v5_clusters.xlsx`, sheet *Screening & equity*), so the diagram updates if you change a decision in the Excel file.

* Records removed before screening (duplicates, automated exclusion with the Table 3 terms) come from the R screening script and are set in cell 3.
* An exclusion counts at the **title/abstract stage** if the full text column is not `Y`, and at the **full-text stage** if it is `Y` (same rule as the *PRISMA 2020* sheet of the workbook).
* Exclusion reasons use the column *Exclusion category (for PRISMA)*.

Texts, positions and colours can be edited in the two **EDIT HERE** cells. The figure is saved as PNG and JPG (300 dpi), PDF and SVG.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib import font_manager

# Latin fonts first, then Chinese-capable fonts (matplotlib falls back glyph by glyph)
LATIN_FONTS = ["Arial", "Helvetica", "DejaVu Sans"]
CHINESE_FONTS = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "Noto Sans CJK TC", "Arial Unicode MS"]
installed = {f.name for f in font_manager.fontManager.ttflist}
use_fonts = [f for f in LATIN_FONTS + CHINESE_FONTS if f in installed]
plt.rcParams["font.family"] = use_fonts
plt.rcParams["axes.unicode_minus"] = False
print("Fonts in use:", use_fonts)
if not [f for f in CHINESE_FONTS if f in installed]:
    print("WARNING: no Chinese-capable font found - Chinese characters will appear as empty boxes.")

In [ ]:
# ===================== Counts from the screening workbook =====================
INPUT = "screening_v5_clusters.xlsx"
SHEET = "Screening & equity"

# From the R screening script (sheet 'PRISMA 2020'): fixed numbers
IDENTIFIED  = 334   # CNKI professional search, academic journals, 28 Oct 2026
DUPLICATES  = 3
AUTOMATED   = 60    # removed by the Table 3 exclusion terms (mitigation lexicon)

df = pd.read_excel(INPUT, sheet_name=SHEET, header=1)
dec = df["YOUR DECISION (Include/Exclude)"].astype(str).str.strip().str.lower()
ft  = df["Full text retrieved (Y/N)"].astype(str).str.strip().str.upper()
cat = df["Exclusion category (for PRISMA)"].astype(str).str.strip()

assert (dec.isin(["include", "exclude"])).all(), "Some records have no Include/Exclude decision"
assert not ((dec == "include") & ~ft.isin(["Y", "N"])).any(), "Some included records have no full-text status (Y/N)"

excl_ta = (dec == "exclude") & (ft != "Y")
excl_ft = (dec == "exclude") & (ft == "Y")
N = dict(
    identified    = IDENTIFIED,
    duplicates    = DUPLICATES,
    automated     = AUTOMATED,
    screened      = len(df),
    excluded_ta   = int(excl_ta.sum()),
    sought        = int((dec == "include").sum() + excl_ft.sum()),
    not_retrieved = int(((dec == "include") & (ft == "N")).sum()),
    excluded_ft   = int(excl_ft.sum()),
    included      = int(((dec == "include") & (ft == "Y")).sum()),
)
N["assessed"] = N["sought"] - N["not_retrieved"]

# Consistency checks: the notebook stops if the flow does not add up
assert N["identified"] - N["duplicates"] - N["automated"] == N["screened"]
assert N["screened"] - N["excluded_ta"] == N["sought"]
assert N["assessed"] - N["excluded_ft"] == N["included"]

# Exclusion reasons (largest first)
REASONS_TA = cat[excl_ta].value_counts()
REASONS_FT = cat[excl_ft].value_counts()
assert REASONS_TA.sum() == N["excluded_ta"] and REASONS_FT.sum() == N["excluded_ft"]
print(N); print(REASONS_TA); print(REASONS_FT)

# Year range of the included corpus (ID 49 is an online-first 2025 article without year in the export)
inc = df[(dec == "include") & (ft == "Y")].copy()
yrs = pd.to_numeric(inc["Year"], errors="coerce").fillna(2025).astype(int)
YEAR_RANGE = f"{yrs.min()}–{yrs.max()}"
print("Included corpus:", YEAR_RANGE)

In [ ]:
# ===================== EDIT HERE (1/2): texts =====================
SHORT = {   # shorter labels for the exclusion categories in the boxes
    "No substantive social/governance dimension": "No social/governance dimension",
    "Not a research article": "Not a research article",
    "Mitigation focus": "Mitigation focus",
    "Not adaptation-focused / off-topic": "Not adaptation-focused / off-topic",
    "Translation / non-Chinese authorship or language": "Translation / not in Chinese",
    "Other (see reason)": "Other",
}
def reason_lines(series):
    return [(f"{i}. {SHORT.get(k, k)} (n = {v})", "item") for i, (k, v) in enumerate(series.items(), 1)]

MAIN = {
    "identified": [
        ("Records identified via CNKI (知网)", "title"),
        ("Professional search (专业检索), title–keyword–abstract", "italic"),
        ("(适应气候变化 OR 气候变化适应) AND", "italic"),
        ("(公平 · 正义 · 社会影响 · 公众参与 · 弱势群体 ·", "italic"),
        ("脆弱群体 · 贫困 · 性别 · 治理)", "italic"),
        ("Academic journals (学术期刊) · searched 28 Oct 2026", "italic"),
        (f"(n = {N['identified']})", "n"),
    ],
    "screened": [
        ("Records screened", "title"),
        ("Title · abstract · keywords (manual)", "italic"),
        ("SPIDER-based eligibility criteria", "italic"),
        ("AI-assisted first-pass suggestions, author decisions", "italic"),
        (f"(n = {N['screened']})", "n"),
    ],
    "sought": [
        ("Reports sought for retrieval", "title"),
        ("CNKI access · library portal · DOI / URL", "italic"),
        (f"(n = {N['sought']})", "n"),
    ],
    "assessed": [
        ("Reports assessed for eligibility", "title"),
        ("Full-text reading (全文评估)", "italic"),
        ("Same criteria as title/abstract screening", "italic"),
        (f"(n = {N['assessed']})", "n"),
    ],
    "included": [
        ("Studies included in review", "title"),
        ("纳入研究 · thematic and equity analysis", "italic"),
        (f"7 thematic clusters · Coverage {YEAR_RANGE}", "italic"),
        ("Policy · Agriculture · Social equity · Urban", "normal"),
        ("Legal · Financing · Science & technology", "normal"),
        (f"(n = {N['included']})", "n"),
    ],
}
SIDE = {
    "removed": [
        ("Records removed before screening:", "title2"),
        (f"Duplicate records (n = {N['duplicates']})", "item"),
        (f"Excluded by automation tool (n = {N['automated']})", "item"),
        ("   R script, mitigation terms (Table 3):", "itemsub"),
        ("   金融 · 双碳 · 碳汇 · 碳中和 · 碳市场 · 碳交易 · 低碳经济", "itemsub"),
    ],
    "excluded_ta": [("Records excluded:", "title")] + reason_lines(REASONS_TA) + [(f"(n = {N['excluded_ta']})", "n")],
    "not_retrieved": [
        ("Reports not retrieved", "title"),
        ("Full text not available", "normal"),
        (f"(n = {N['not_retrieved']})", "n"),
    ],
    "excluded_ft": [("Reports excluded:", "title")] + reason_lines(REASONS_FT) + [(f"(n = {N['excluded_ft']})", "n")],
}
HEADER = "Identification of studies via databases"
FOOTER = "PRISMA 2020 — Page MJ et al. BMJ 2021;372:n71. doi: 10.1136/bmj.n71"

In [ ]:
# ===================== EDIT HERE (2/2): positions, sizes, colours =====================
# Canvas in pixels (figure = W/100 x H/100 inches). y grows DOWNWARDS.
W, H = 900, 910
COL = dict(header_face="#E8A020", header_edge="#C98A1A", bar="#6395CB",
           box_face="white", box_edge="#2B2B2B", sub="#555555",
           arrow="#2B2B2B", arrow_last="#9A9A9A", sep="#CFCFCF", footer="#8A8A8A")

MX0, MX1 = 94, 450     # main column
SX0, SX1 = 490, 870    # side column

MAIN_POS = {
    "identified": (MX0,  90, MX1, 245),
    "screened":   (MX0, 280, MX1, 425),
    "sought":     (MX0, 450, MX1, 525),
    "assessed":   (MX0, 550, MX1, 695),
    "included":   (MX0, 735, MX1, 865),
}
SIDE_POS = {
    "removed":       (SX0,  90, SX1, 245),
    "excluded_ta":   (SX0, 280, SX1, 425),
    "not_retrieved": (SX0, 450, SX1, 525),
    "excluded_ft":   (SX0, 550, SX1, 695),
}
H_ARROWS = [("identified", "removed"), ("screened", "excluded_ta"),
            ("sought", "not_retrieved"), ("assessed", "excluded_ft")]
PHASES = [("Identification", 90, 245), ("Screening", 280, 695), ("Included", 735, 865)]
BAR_X0, BAR_X1 = 27, 80
SEPARATORS = [262, 715]
SEP_X0, SEP_X1 = 27, 870

STYLE = dict(
    title   = dict(size=11,  weight="bold",   style="normal", color="black", step=21),
    title2  = dict(size=10.5, weight="bold",  style="normal", color="black", step=24, ha="left"),
    italic  = dict(size=8.5, weight="normal", style="italic", color=COL["sub"], step=16),
    normal  = dict(size=9.5, weight="normal", style="normal", color="black", step=17),
    item    = dict(size=9.5, weight="normal", style="normal", color="black", step=18, ha="left"),
    itemsub = dict(size=8.5, weight="normal", style="italic", color=COL["sub"], step=16, ha="left"),
    n       = dict(size=11,  weight="bold",   style="normal", color="black", step=0),
)
TOP_PAD = 19
N_FROM_BOTTOM = 15
BOX_LW = 1.4

In [ ]:
def draw_box(ax, pos, lines):
    x0, y0, x1, y1 = pos
    ax.add_patch(FancyBboxPatch((x0, y0), x1 - x0, y1 - y0, boxstyle="round,pad=0,rounding_size=7",
                                fc=COL["box_face"], ec=COL["box_edge"], lw=BOX_LW, zorder=2))
    cx, y = (x0 + x1) / 2, y0 + TOP_PAD
    for text, st in lines:
        s = STYLE[st]
        if st == "n":
            ax.text(cx, y1 - N_FROM_BOTTOM, text, ha="center", va="center", fontsize=s["size"],
                    fontweight=s["weight"], zorder=3)
            continue
        ha = s.get("ha", "center")
        ax.text(x0 + 14 if ha == "left" else cx, y, text, ha=ha, va="center", fontsize=s["size"],
                fontweight=s["weight"], fontstyle=s["style"], color=s["color"], zorder=3)
        y += s["step"]


def arrow(ax, p_from, p_to, color):
    ax.annotate("", xy=p_to, xytext=p_from, zorder=1,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.3, shrinkA=0, shrinkB=0, mutation_scale=11))


def build_figure():
    fig = plt.figure(figsize=(W / 100, H / 100), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.axis("off")
    ax.add_patch(FancyBboxPatch((22, 12), SX1 - 22, 50, boxstyle="round,pad=0,rounding_size=7",
                                fc=COL["header_face"], ec=COL["header_edge"], lw=1.2, zorder=2))
    ax.text((22 + SX1) / 2, 37, HEADER, ha="center", va="center", fontsize=13, fontweight="bold", zorder=3)
    for label, ya, yb in PHASES:
        ax.add_patch(FancyBboxPatch((BAR_X0, ya), BAR_X1 - BAR_X0, yb - ya, boxstyle="round,pad=0,rounding_size=7",
                                    fc=COL["bar"], ec="none", zorder=2))
        ax.text((BAR_X0 + BAR_X1) / 2, (ya + yb) / 2, label, rotation=90, ha="center", va="center",
                color="white", fontsize=11, fontweight="bold", zorder=3)
    for y in SEPARATORS:
        ax.plot([SEP_X0, SEP_X1], [y, y], ls=(0, (4, 3)), color=COL["sep"], lw=1.2, zorder=1)
    for k, pos in MAIN_POS.items():
        draw_box(ax, pos, MAIN[k])
    for k, pos in SIDE_POS.items():
        draw_box(ax, pos, SIDE[k])
    keys = list(MAIN_POS)
    for i in range(len(keys) - 1):
        a, b = MAIN_POS[keys[i]], MAIN_POS[keys[i + 1]]
        xm = (a[0] + a[2]) / 2
        arrow(ax, (xm, a[3]), (xm, b[1]), COL["arrow_last"] if i == len(keys) - 2 else COL["arrow"])
    for m, s in H_ARROWS:
        a, b = MAIN_POS[m], SIDE_POS[s]
        arrow(ax, (a[2], (a[1] + a[3]) / 2), (b[0], (a[1] + a[3]) / 2), COL["arrow"])
    ax.text(W / 2, H - 22, FOOTER, ha="center", va="center", fontsize=8, color=COL["footer"])
    return fig

In [ ]:
fig = build_figure()
OUT = "PRISMA_flow_diagram_96"
fig.savefig(OUT + ".png", dpi=300)
fig.savefig(OUT + ".jpg", dpi=300, pil_kwargs={"quality": 95})
fig.savefig(OUT + ".pdf")
fig.savefig(OUT + ".svg")
plt.show()

## Quick tips
- **Numbers** come from the Excel file: change a decision there and re-run. The checks stop the notebook if the flow does not add up or if a record has no decision / no full-text status.
- **Texts**: edit `MAIN` / `SIDE` (cell 4). One tuple = one line.
- **Box sizes and positions**: `MAIN_POS`, `SIDE_POS`, `PHASES`, `SEPARATORS` (cell 5). If you add lines, make the box taller (4th value = bottom edge).
- If Chinese characters show as empty boxes, add an installed Chinese font to `CHINESE_FONTS` (cell 2).